In [70]:
spark

In [69]:
%run /opt/spark-notebooks/silver_base/parsed_data.ipynb


Master DataFrame 'parsed_df' successfully created and listening to Bronze!


In [71]:
from pyspark.sql.functions import col, explode
# 1. Create the database table mapping
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_base.discount_fact_RT (
        order_id STRING,
        item_id STRING,
        discount_type STRING,
        code STRING,
        amount DECIMAL(10,2),
        source_system STRING,
        bronze_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/base/discount_fact_RT'
""")

DataFrame[]

In [72]:
items_for_discount_df = parsed_df.select(
    col("data.order.order_id").alias("order_id"),
    explode(col("data.order.items")).alias("item"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [73]:
discounts_exploded_df = items_for_discount_df.select(
    col("order_id"),
    col("item.item_id").alias("item_id"),
    explode(col("item.discounts")).alias("discount"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [74]:
silver_discount_fact_df = discounts_exploded_df.select(
    col("order_id"),
    col("item_id"),
    col("discount.discount_type").alias("discount_type"),
    col("discount.code").alias("code"),
    col("discount.amount").cast("decimal(10,2)").alias("amount"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [75]:
discount_fact_query = (
    silver_discount_fact_df.writeStream
    .format("delta")
    .queryName("silver_discount_fact_stream")
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark-data/checkpoints/silver/base/discount_fact_RT/v1")
    .trigger(processingTime="30 seconds")
    .start("/opt/spark-data/delta/silver/base/discount_fact_RT")
)
print("Started silver_discount_fact_RT continuous stream! 🏷️")

Started silver_discount_fact_RT continuous stream! 🏷️


In [77]:
spark.sql("SELECT * FROM silver_base.discount_fact_RT").limit(20).toPandas()


,order_id,item_id,discount_type,code,amount,source_system,bronze_ingest_ts
0,ORD-10005,ITEM-10005,ITEM_PROMO,MOBILE5,2000.00,order_service,2026-03-27 16:27:10.891
1,ORD-10001,ITEM-10001,ITEM_PROMO,ELEC15,3000.00,order_service,2026-03-27 16:27:10.891
